In [3]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)
from tqdm import tqdm

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [5]:
from google.colab import files

uploaded = files.upload()

Saving train.csv to train.csv


In [86]:
df = pd.read_csv("train.csv")

df = df[df["lang_abv"] != "en"].copy()

print("Samples:", len(df))

Samples: 5250


In [87]:
print(df["label"].value_counts())

label
2    1787
0    1749
1    1714
Name: count, dtype: int64


In [93]:
MODEL_NAME = "microsoft/mdeberta-v3-base"

MAX_LEN = 90

In [92]:
# lengths = []

# for _, row in df.iterrows():
#     tokens = tokenizer(
#         row["premise"],
#         row["hypothesis"]
#     )["input_ids"]

#     lengths.append(len(tokens))

# print(pd.Series(lengths).quantile(0.95))

97.0


In [103]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [94]:
class NLIDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_len, has_labels=True):
        self.df = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_labels = has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        premise = str(self.df.loc[idx, "premise"])
        hypothesis = str(self.df.loc[idx, "hypothesis"])

        encoding = self.tokenizer(
            premise,
            hypothesis,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt"
        )

        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
        }

        if self.has_labels:
            item["label"] = torch.tensor(
                int(self.df.loc[idx, "label"]),
                dtype=torch.long
            )

        return item

dataset = NLIDataset(
    df,
    tokenizer,
    MAX_LEN,
    has_labels=True
)

In [95]:
BATCH_SIZE = 16

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

generator = torch.Generator().manual_seed(42)

train_dataset, val_dataset = random_split(
    dataset,
    [train_size, val_size],
    generator=generator
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train:", len(train_dataset))
print("Val:", len(val_dataset))

Train: 4200
Val: 1050


In [96]:
EPOCHS = 4

In [104]:
LR = 2e-5

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    torch_dtype=torch.float32
)

for layer in model.deberta.encoder.layer[:4]:
    for param in layer.parameters():
        param.requires_grad = False

model.to(device)

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/mdeberta-v3-base
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.weight         | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.weight                        | MISSING    | 
classifier.weight   

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

In [105]:
optimizer = AdamW(
    model.parameters(),
    weight_decay=0.01,
    lr=LR
)

from transformers import get_linear_schedule_with_warmup

num_training_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps
)

In [106]:
def accuracy(logits, labels):

    preds = torch.argmax(logits, dim=1)

    correct = (preds == labels).sum().item()

    return correct, labels.size(0)

In [107]:
def train_epoch(model, loader):

    model.train()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    progress = tqdm(loader)

    for batch in progress:

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        scheduler.step()

        total_loss += loss.item()

        correct, total = accuracy(logits, labels)

        total_correct += correct
        total_samples += total

        progress.set_postfix(
            loss=loss.item()
        )

    avg_loss = total_loss / len(loader)

    avg_acc = 100 * total_correct / total_samples

    return avg_loss, avg_acc


In [108]:
def validate(model, loader):

    model.eval()

    total_loss = 0
    total_correct = 0
    total_samples = 0

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()

            correct, total = accuracy(logits, labels)

            total_correct += correct
            total_samples += total

    avg_loss = total_loss / len(loader)

    avg_acc = 100 * total_correct / total_samples

    return avg_loss, avg_acc

In [109]:
best_val_acc = 0

history = []

for epoch in range(EPOCHS):

    train_loss, train_acc = train_epoch(
        model,
        train_loader
    )

    val_loss, val_acc = validate(
        model,
        val_loader
    )

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc
    })

    print("\n")
    print(f"Epoch {epoch+1}/{EPOCHS}")
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.2f}%")
    print(f"Val Loss  : {val_loss:.4f}")
    print(f"Val Acc   : {val_acc:.2f}%")
    print("-" * 50)

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_nli_model.pth"
        )

print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

history_df = pd.DataFrame(history)

print(history_df)

100%|██████████| 263/263 [01:39<00:00,  2.66it/s, loss=0.861]




Epoch 1/4
Train Loss: 1.0711
Train Acc : 39.69%
Val Loss  : 0.8563
Val Acc   : 63.14%
--------------------------------------------------


100%|██████████| 263/263 [01:38<00:00,  2.66it/s, loss=0.53]




Epoch 2/4
Train Loss: 0.7163
Train Acc : 70.26%
Val Loss  : 0.7162
Val Acc   : 71.90%
--------------------------------------------------


100%|██████████| 263/263 [01:39<00:00,  2.65it/s, loss=0.48]




Epoch 3/4
Train Loss: 0.4487
Train Acc : 83.62%
Val Loss  : 0.7399
Val Acc   : 73.14%
--------------------------------------------------


100%|██████████| 263/263 [01:39<00:00,  2.66it/s, loss=0.237]




Epoch 4/4
Train Loss: 0.2975
Train Acc : 90.05%
Val Loss  : 0.7859
Val Acc   : 74.57%
--------------------------------------------------
Best Validation Accuracy: 74.57%
   epoch  train_loss  train_acc  val_loss    val_acc
0      1    1.071105  39.690476  0.856298  63.142857
1      2    0.716331  70.261905  0.716245  71.904762
2      3    0.448674  83.619048  0.739918  73.142857
3      4    0.297472  90.047619  0.785857  74.571429


In [110]:
from google.colab import files

uploaded = files.upload()

In [111]:
tf = pd.read_csv('test.csv')

In [112]:
tf = tf[tf['lang_abv'] != 'en']

In [113]:
dataset = NLIDataset(
    tf,
    tokenizer,
    MAX_LEN,
    has_labels=False
)

In [114]:
print(len(dataset))

2250


In [115]:
test_loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [116]:
model.eval()

op = []
with torch.no_grad():

  for batch in test_loader:

      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      # labels = batch["label"].to(device)

      outputs = model(
          input_ids=input_ids,
          attention_mask=attention_mask,
          # labels=labels
      )

      preds = torch.argmax(outputs.logits, dim=1)

      op.extend(
          preds.cpu().numpy().tolist()
      )

In [117]:
tf['prediction'] = op

In [118]:
tf[['id','prediction']].to_csv('predictions_mul.csv',index=False)

In [119]:
ef = pd.read_csv('predictions_eng.csv')

In [120]:
result_df = pd.concat([tf[['id','prediction']], ef], ignore_index=True)

In [125]:
df = pd.read_csv("test.csv")

In [127]:
result_df = result_df.reindex(df.index)

In [128]:
result_df.to_csv('submission.csv')